<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [25]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q
!pip install tqdm -q

# First-party imports
import os
import json
import csv
import time
import random
from typing import List, Dict, Any, Optional, Tuple
import numpy as np # Import numpy

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from tqdm.notebook import tqdm
from google.colab import drive

In [26]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured llm review directory exists at: {LLM_DIR}.")

# Define and create RES_DIR for the result file
RES_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(RES_DIR, exist_ok=True)
print(f"Ensured result directory exists at: {RES_DIR}.")

# Define and create RES_DIR for the log file
LOG_DIR = os.path.join(WORKING_DIR, "tvd_mi")
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Ensured logging directory exists at: {LOG_DIR}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/iclr/final.
Ensured llm review directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.
Ensured result directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi.
Ensured logging directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi.


In [27]:
# @title Definitions

# Target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Model
MODEL = "gpt-4o-mini-2024-07-18" # Robertson and Kayejo only specified "gpt-4o-mini"

# Hyperparameters
TEMPERATURE = 0.0 # As Robertson and Koyejo (2025) (in their repo)
MAX_TOKENS = 2000 # As Robertson and Koyejo (2025) (in their repo)
MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random

# API Keys
#OPENAI_KEY = google.colab.userdata.get('OPENAI_API_KEY')

# Define the type of TVD-MI score to calculate using a numerical option
# 1: "paper / human review"
# 2: "paper / llm review"
# 3: "human review / llm review"
# 4: "llm review / llm review"
TVD_MI_SCORE_OPTION = 2

# Define parameter columns for grouping and logging
PARAM_COLS = ["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

# Define log file columns
LOG_COL = ['combo', 'TPR_calculated', 'FPR_calculated', 'complete']

# Define result file columns
RES_COL = [
    'paper_id',
] + PARAM_COLS + [
    'tpr_response_a', 'tpr_response_b', 'tpr_tvd_mi_score_1', 'tpr_tvd_mi_score_2', 'tpr_avg_tvd_mi_score',
    'fpr_response_a', 'fpr_response_b', 'fpr_tvd_mi_score_1', 'fpr_tvd_mi_score_2', 'fpr_avg_tvd_mi_score'
]

SEED_VALUE = 7 # Random value

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

# Set to a positive integer to limit the total number of rows processed per group. Set to 0 or a negative value to process all available rows per group.
NUM_ROWS_PER_GROUP = 98

SIM_SKIP_PROBABILITY = 0.1 # Probability of simulating a skip (for simulation mode only)

TOTAL_FAILURE_MESSAGE = "LLM call failed after multiple retries"

SKIP_MESSAGE = "Skipped cause of invalid input"

# Custom strings that should trigger a skip in TVD-MI score calculation
SPECIAL_SKIP_STRINGS = {
    "ERROR: NOTE TAKING NOT SUCCESSFUL",
    "ERROR: REVIEW GENERATION NOT SUCCESSFUL",
    "ERROR: UNKNOWN STATE REACHED",
    "ERROR: Final response key missing from JSON.",
    "NOTE TAKING NOT ATTEMPTED"
}

In [37]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target columns for paper content and human reviews
paper_content = dataset_paper[['paper_id', 'parsed_text', 'human_review_1', 'human_review_2', 'human_review_3']].copy()

# Remove metadata from the papers full text (ensure no shortcut identification for TVD-MI exists)
paper_content['parsed_text'] = paper_content['parsed_text'].str.partition(HEADER_TO_REMOVE)[2]

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

# Read llm reviews
dataset_llm = pd.read_parquet(os.path.join(LLM_DIR, "llm_reviews_results.parquet"))

# Check dataset_llm
display(dataset_llm.head(3))
display(dataset_llm.shape)

,paper_id,parsed_text,human_review_1,human_review_2,human_review_3
0,vuD2xEtxZcj,"In deep learning, fine-grained N:M sparsity r...",Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...
1,WoByU5W5te0,We present a novel framework to regularize Ne...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focus on t...,Summary Of The Paper:\n\nThe paper proposes a ...
2,XZRmNjUMj0c,Neural Architecture Search (NAS) is a fast-de...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focused on...,Summary Of The Paper:\n\nThe paper introduces ...


(98, 5)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes_1,llm_review_1,parsed_llm_review_1
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,FAITHFUL,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,,FAITHFUL,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW
2,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,OBJECTIVE,This is a simulated note based on: Model='gpt-...,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...",PLACEHOLDER REVIEW


(416, 9)

In [38]:
print("Shape of dataset_llm:")
display(dataset_llm.shape)

Shape of dataset_llm:


(416, 9)

In [39]:
print("Shape of merged_df:")
display(merged_df.shape)

Shape of merged_df:


(512, 13)

In [29]:
# @title Merge data

merged_df = pd.merge(paper_content, dataset_llm, on='paper_id', how='inner')
display(merged_df.head(3))
display(merged_df.shape)

,paper_id,parsed_text,human_review_1,human_review_2,human_review_3,model,temperature,reasoning_effort,chain_of_thought,note_taking,llm_notes_1,llm_review_1,parsed_llm_review_1
0,-Y34L45JR6z,Pre-training with offline data and online fin...,Summary Of The Paper:\n\nThe paper addresses h...,Summary Of The Paper:\n\nThis paper presents a...,Summary Of The Paper:\n\nThe paper considers t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0WVNuEnqVu,Dynamic diagnosis is desirable when medical t...,Summary Of The Paper:\n\nThis paper investigat...,Summary Of The Paper:\n\nThe paper proposed a ...,Summary Of The Paper:\n\nThis paper studies se...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1Wo0vqaZ8WJ,Offline reinforcement learning aims to train ...,Summary Of The Paper:\n\nThe paper utilized *N...,Summary Of The Paper:\n\nThis paper presents a...,Summary Of The Paper:\n\nOffline RL is challen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(512, 13)

### API

In [30]:
def generate_tvd_mi_prompt(response_a: str, response_b: str) -> str:
    """Generate prompt for TVD-MI critic"""
    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: Scientific paper peer review task

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)"""

    return prompt

class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7):
        """Initialize LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)
        self.cache = {}

        # Placeholder for real when simulation=False:
        self._openai = None

    def _maybe_init_clients(self):
        """Start simulation or API clients"""
        # Simulation
        if self.simulate:
            return
        # Initialize API client
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)

    def call(
        self,
        prompt: str,
        model: str = MODEL,
        temperature: float = TEMPERATURE,
        max_tokens: int = MAX_TOKENS,
        max_retries: int = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
    ) -> Tuple[str, Dict[str, Any]]:
        """
        Define the API calls / simulations according to the varying or hardcoded input.
        Judge if content paris are from the same source and state the confidence fo judgement.

        """
        self._maybe_init_clients()

        # In simulation
        if self.simulate:

            # Create artificial response options
            possible_responses = [
                "[significant gain]",
                "[little gain]",
                "[no gain]",
                "[total failure]"
            ]

            # Randomly choose one option
            simulated_response = self.rng.choice(possible_responses)

            # Construct artificial raw output
            raw = {
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "provider": "simulated",
                "response": simulated_response
            }

            # Make the response human readable
            structured = json.dumps(raw, indent=2)

            return structured, raw

        # In active mode try successful calls till max_retries
        for attempt in range(1, max_retries + 1):
            try:

                # Create result items
                response_text = ""
                provider = ""

                resp = self._openai.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    max_tokens=max_tokens
                )
                response_text = resp.choices[0].message.content
                provider = "openai"

                # Create full response
                raw = {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "provider": provider,
                    "response": response_text
                }

                # Make the response human readable
                structured = json.dumps(raw, indent=2)

                return structured, raw

            except Exception as e:
                print(f"[LLM ERROR]: {e}", flush = True)
                if attempt == max_retries:
                  # Create error in case of failure till max_retries
                  error_raw = {"status": "error", "model": model, "response": "[total failure]"}

                  # Make error human readable
                  return json.dumps(error_raw, indent=2), error_raw

                time.sleep(retry_delay)

def interpret_tvd_mi_response(response: str) -> str:
    """Convert LLM response to numeric score (as string) or return specific message strings."""

    # Standardize LLM response
    response = response.strip().lower()

    if "[significant gain]" in response:
        return "1.0"
    elif "[little gain]" in response:
        return "0.25"
    elif "[no gain]" in response:
        return "0.0"
    elif "[total failure]" in response:
        return TOTAL_FAILURE_MESSAGE
    else:
        # Default to no gain if response is unclear, and return as string
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return "0.0"

### Helpers for logging, status output and keys

In [31]:
# Helper to handle NaN values when creating hashable keys for comparison
def _nan_to_none(x):
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

# Function to create a unique hashable key for a paper-parameter combination
def get_unique_combo_key(row, param_cols):
    key_elements = [row['paper_id']]
    for col in param_cols:
        key_elements.append(_nan_to_none(row[col]))
    return tuple(key_elements)

# Helper to format parameter string for logging and printing
def format_param_strings(paper_id_val, param_dict):
    # For log file: paper_id=X|model=Y|temp=Z...
    log_parts = [f"paper_id={paper_id_val}"]
    for col in PARAM_COLS:
        value = param_dict.get(col)
        log_parts.append(f"{col}={_nan_to_none(value)}")
    combo_str = "|".join(log_parts)

    # For print message: model=Y, temperature=Z...
    # Note: paper_id will be prefixed separately for printing
    print_param_parts = []
    for col in PARAM_COLS:
        value = param_dict.get(col)
        print_param_parts.append(f"{col}={'' if pd.isna(value) else value}")
    print_params_str = ", ".join(print_param_parts)

    return combo_str, print_params_str

# Helper to parse values from the combo string to their original values
def _parse_val_from_log_str(val_str):
    if val_str == 'None':
        return None
    try:
        # Attempt float conversion for numbers (e.g., temperature)
        return float(val_str)
    except ValueError:
        return val_str # If not a float, return as string (e.g., 'low', 'abstract')

# Function to create a unique hashable key from the combo string for log file parsing
def get_unique_combo_key_from_log_entry(combo_str_from_log, param_cols):
    key_elements = []
    parts = combo_str_from_log.split('|')

    # Extract paper_id from the first part
    paper_id_part = parts[0]
    paper_id = paper_id_part.split('=', 1)[1]
    key_elements.append(paper_id)

    # Extract parameter values from the remaining parts
    param_map = {}
    for part in parts[1:]:
        key, val_str = part.split('=', 1)
        param_map[key] = _parse_val_from_log_str(val_str)

    # Assemble key_elements in the same order as get_unique_combo_key expects
    for col in param_cols:
        key_elements.append(param_map.get(col)) # Use .get to safely handle cases where a param might be missing

    return tuple(key_elements)

# Helper function to check if a string represents a numeric value
def is_numeric_string(s):
    if not isinstance(s, str):
        return False
    try:
        float(s)
        return True
    except ValueError:
        return False

### Helper for TVD-MI score type settings

In [32]:
def configure_tvd_mi_run(option: int, res_dir: str, log_dir: str) -> Tuple[List[str], str, str]:
    """
    Configures the column pair and output paths based on the TVD-MI score option.
    Randomly selects human/LLM review column if multiple are available.
    """
    column_pair = []
    file_suffix = ""
    score_type_label = ""

    # Dynamically get all human review columns
    all_human_review_cols = [col for col in paper_content.columns if col.startswith('human_review_')]
    # Dynamically get all LLM review columns
    all_llm_review_cols = [col for col in dataset_llm.columns if col.startswith('parsed_llm_review_')]

    # Randomly select one human review column if available
    selected_human_review_col = random.choice(all_human_review_cols) if all_human_review_cols else ""
    # Randomly select one LLM review column if available
    selected_llm_review_col = random.choice(all_llm_review_cols) if all_llm_review_cols else ""

    if option == 1:
        score_type_label = "paper / human review"
        if not selected_human_review_col:
            raise ValueError("No human review columns available for 'paper / human review' comparison.")
        column_pair = ["parsed_text", selected_human_review_col]
        file_suffix = f"paper_{selected_human_review_col}"

    elif option == 2:
        score_type_label = "paper / llm review"
        if not selected_llm_review_col:
            raise ValueError("No LLM review columns available for 'paper / llm review' comparison.")
        column_pair = ["parsed_text", selected_llm_review_col]
        file_suffix = f"paper_{selected_llm_review_col}"

    elif option == 3:
        score_type_label = "human review / llm review"
        if not selected_human_review_col or not selected_llm_review_col:
            raise ValueError("Not enough human/LLM review columns available for 'human review / llm review' comparison.")
        column_pair = [selected_human_review_col, selected_llm_review_col]
        file_suffix = f"h_{selected_human_review_col}_l_{selected_llm_review_col}"

    elif option == 4:
        score_type_label = "llm review / llm review"
        # Check if there are sufficient llm reviews for this case
        if len(all_llm_review_cols) < 2:
            raise ValueError("Not enough LLM review columns available for 'llm review / llm review' comparison (need at least two distinct ones).")

        # Select the first LLM review column
        selected_llm_review_col_1 = selected_llm_review_col

        # Select a second, *different* LLM review column
        remaining_llm_review_cols = [col for col in all_llm_review_cols if col != selected_llm_review_col_1]
        selected_llm_review_col_2 = random.choice(remaining_llm_review_cols)

        column_pair = [selected_llm_review_col_1, selected_llm_review_col_2]
        file_suffix = f"llm_{selected_llm_review_col_1}_llm_{selected_llm_review_col_2}"
    else:
        raise ValueError(f"Invalid TVD_MI_OPTION: {option}. Please choose between 1 and 4.")

    # Create output and log paths
    output_path = os.path.join(res_dir, f"tvd_mi_scores_{file_suffix}.parquet")
    log_path = os.path.join(log_dir, f"tvd_mi_log_{file_suffix}.parquet")

    return column_pair, output_path, log_path

### Higher order Functions

In [33]:
def calculate_tvd_mi_scores_for_pair(llm_client, response_a_text, response_b_text):
    """
    Generates prompts, calls simulations / API clients, and interprets scores for a given pair of responses.
    All processes are balanced and the resulting scores are averaged before returning.
    """
    # --- A, B ---

    # Generate the TVD-MI prompt
    tvd_mi_prompt_1 = generate_tvd_mi_prompt(response_a_text, response_b_text)

    # Call the LLMClient
    structured_response_1, raw_response_1 = llm_client.call(
        prompt=tvd_mi_prompt_1,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    # Interpret the LLM's response to get the TVD-MI score (as string)
    llm_critic_response_text_1 = raw_response_1.get('response')
    tvd_mi_score_1_str = interpret_tvd_mi_response(llm_critic_response_text_1)

    # --- B, A ---

    # Generate the TVD-MI prompt
    tvd_mi_prompt_2 = generate_tvd_mi_prompt(response_b_text, response_a_text)

    # Call the LLMClient
    structured_response_2, raw_response_2 = llm_client.call(
        prompt=tvd_mi_prompt_2,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    # Interpret the LLM's response to get the TVD-MI score (as string)
    llm_critic_response_text_2 = raw_response_2.get('response')
    tvd_mi_score_2_str = interpret_tvd_mi_response(llm_critic_response_text_2)

    # --- Results ---

    # Calculate the average TVD-MI score, handling potential string failures
    if tvd_mi_score_1_str == TOTAL_FAILURE_MESSAGE or tvd_mi_score_2_str == TOTAL_FAILURE_MESSAGE:
        avg_tvd_mi_score_str = TOTAL_FAILURE_MESSAGE
    elif tvd_mi_score_1_str == SKIP_MESSAGE or tvd_mi_score_2_str == SKIP_MESSAGE:
        avg_tvd_mi_score_str = SKIP_MESSAGE
    else:
        # Both are expected to be numeric strings here, convert to float for calculation
        try:
            avg_tvd_mi_score = (float(tvd_mi_score_1_str) + float(tvd_mi_score_2_str)) / 2
            avg_tvd_mi_score_str = str(avg_tvd_mi_score)
        except ValueError:
            avg_tvd_mi_score_str = TOTAL_FAILURE_MESSAGE

    return {
        'response_a': response_a_text,
        'response_b': response_b_text,
        'prompt_1': tvd_mi_prompt_1,
        'raw_response_1': json.dumps(raw_response_1),
        'structured_response_1': structured_response_1,
        'prompt_2': tvd_mi_prompt_2,
        'raw_response_2': json.dumps(raw_response_2),
        'structured_response_2': structured_response_2,
        'tvd_mi_score_1': tvd_mi_score_1_str,
        'tvd_mi_score_2': tvd_mi_score_2_str,
        'avg_tvd_mi_score': avg_tvd_mi_score_str
    }

In [34]:
def process_single_paper(llm_client, paper_id, row, response_a_col_name, response_b_col_name, merged_df, param_cols):
    """
    Process a single paper_id, calculating TPR and FPR scores and returning them.
    Return a tuple of (results_data_dict, log_data_dict).
    """
    # --- Initialize results for single row in result file ---
    data_to_write = {'paper_id': paper_id}

    # Add parameter values from the current row to the results dictionary
    param_dict = {}
    for param_col in param_cols:
        param_dict[param_col] = row[param_col]
        data_to_write[param_col] = row[param_col]

    # --- TPR Calculation: response_a and response_b from the current row ---

    # Extract responses from the current rows
    response_a_tpr_text = str(row[response_a_col_name]) if pd.notna(row[response_a_col_name]) else ""
    response_b_tpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    # Introduce random skip for simulation mode
    should_simulated_skip_tpr = SIMULATION_ACTIVE and llm_client.rng.random() < SIM_SKIP_PROBABILITY

    # Check for skip conditions: simulated skip, or either response is empty/contains a special skip string
    if should_simulated_skip_tpr or (not response_a_tpr_text or response_a_tpr_text in SPECIAL_SKIP_STRINGS) or (not response_b_tpr_text or response_b_tpr_text in SPECIAL_SKIP_STRINGS):
        # Assign the actual (potentially special) response strings, and mark scores as skipped
        data_to_write['tpr_response_a'] = response_a_tpr_text
        data_to_write['tpr_response_b'] = response_b_tpr_text
        data_to_write['tpr_tvd_mi_score_1'] = SKIP_MESSAGE
        data_to_write['tpr_tvd_mi_score_2'] = SKIP_MESSAGE
        data_to_write['tpr_avg_tvd_mi_score'] = SKIP_MESSAGE
    else:
        # Call function for score calculation
        tpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, response_a_tpr_text, response_b_tpr_text)

        # Add TPR results with 'tpr_' prefix
        data_to_write['tpr_response_a'] = tpr_results_raw['response_a']
        data_to_write['tpr_response_b'] = tpr_results_raw['response_b']
        data_to_write['tpr_tvd_mi_score_1'] = tpr_results_raw['tvd_mi_score_1']
        data_to_write['tpr_tvd_mi_score_2'] = tpr_results_raw['tvd_mi_score_2']
        data_to_write['tpr_avg_tvd_mi_score'] = tpr_results_raw['avg_tvd_mi_score']

    # --- FPR Calculation: response_b from current row, response_a from random different row ---

    # Extract response_b from current row
    response_b_fpr_text = str(row[response_b_col_name]) if pd.notna(row[response_b_col_name]) else ""

    # Get all possible indices from the full merged_df, excluding the current row's index 'i'
    eligible_indices_for_a = merged_df.index.drop(row.name).tolist()

    # If no other response_a is available take empty string (should not happen)
    if not eligible_indices_for_a:
        response_a_fpr_text = ""
    # Else, use the new, different response_a
    else:
        random_a_idx = random.choice(eligible_indices_for_a)
        response_a_fpr_text = str(merged_df.loc[random_a_idx, response_a_col_name]) if pd.notna(merged_df.loc[random_a_idx, response_a_col_name]) else ""

    # Introduce random skip for simulation mode
    should_simulated_skip_fpr = SIMULATION_ACTIVE and llm_client.rng.random() < SIM_SKIP_PROBABILITY

    # Check for skip conditions: simulated skip, or either response is empty/contains a special skip string
    if should_simulated_skip_fpr or (not response_a_fpr_text or response_a_fpr_text in SPECIAL_SKIP_STRINGS) or (not response_b_fpr_text or response_b_fpr_text in SPECIAL_SKIP_STRINGS):
        # Assign the actual (potentially special) response strings, and mark scores as skipped
        data_to_write['fpr_response_a'] = response_a_fpr_text
        data_to_write['fpr_response_b'] = response_b_fpr_text
        data_to_write['fpr_tvd_mi_score_1'] = SKIP_MESSAGE
        data_to_write['fpr_tvd_mi_score_2'] = SKIP_MESSAGE
        data_to_write['fpr_avg_tvd_mi_score'] = SKIP_MESSAGE
    else:
        # Call function for score calculation
        fpr_results_raw = calculate_tvd_mi_scores_for_pair(llm_client, response_a_fpr_text, response_b_fpr_text)

        # Add FPR results with 'fpr_' prefix
        data_to_write['fpr_response_a'] = fpr_results_raw['response_a']
        data_to_write['fpr_response_b'] = fpr_results_raw['response_b']
        data_to_write['fpr_tvd_mi_score_1'] = fpr_results_raw['tvd_mi_score_1']
        data_to_write['fpr_tvd_mi_score_2'] = fpr_results_raw['tvd_mi_score_2']
        data_to_write['fpr_avg_tvd_mi_score'] = fpr_results_raw['avg_tvd_mi_score']

    # --- Determine completion status for logging ---
    # A calculation is considered 'calculated' for logging if its average score is a string representation of a number, SKIP_MESSAGE, or TOTAL_FAILURE_MESSAGE
    tpr_calculated = (
        is_numeric_string(data_to_write['tpr_avg_tvd_mi_score']) or
        data_to_write['tpr_avg_tvd_mi_score'] == SKIP_MESSAGE or
        data_to_write['tpr_avg_tvd_mi_score'] == TOTAL_FAILURE_MESSAGE
    )
    fpr_calculated = (
        is_numeric_string(data_to_write['fpr_avg_tvd_mi_score']) or
        data_to_write['fpr_avg_tvd_mi_score'] == SKIP_MESSAGE or
        data_to_write['fpr_avg_tvd_mi_score'] == TOTAL_FAILURE_MESSAGE
    )

    # --- Logging ---

    # Log success states
    log_data_to_write = {
        'TPR_calculated': tpr_calculated,
        'FPR_calculated': fpr_calculated,
        'complete': tpr_calculated and fpr_calculated
    }

    # Format the combo string for logging
    combo_str, _ = format_param_strings(paper_id, param_dict)
    log_data_to_write['combo'] = combo_str

    return data_to_write, log_data_to_write

### TVD-MI score calculation

In [35]:
if __name__ == "__main__":

    # Set seed for reproducibility for the global random module
    random.seed(SEED_VALUE)

    # Configure the run based on the TVD_MI_SCORE_OPTION
    column_pair, result_path, log_path = configure_tvd_mi_run(TVD_MI_SCORE_OPTION, RES_DIR, LOG_DIR)

    # Select prompt input columns based on column_pair
    response_a_col_name = column_pair[0]
    response_b_col_name = column_pair[1]

    # Information output
    print(f"Prompt input will use columns {response_a_col_name} and {response_b_col_name}", flush = True)
    print(f"Processing and saving results to: {result_path}", flush = True)
    print(f"Logging successful iterations to: {log_path}", flush = True)

    # Activate/Deactivate simulation/API calls
    llm_client = LLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

    # Determine the DataFrame to iterate over and the actual parameters for this run
    df_to_use_for_iteration = None
    current_param_cols_for_run = []

    if TVD_MI_SCORE_OPTION == 1:
        # For paper/human review, no LLM-specific parameters apply for grouping
        df_to_use_for_iteration = paper_content.copy()
        current_param_cols_for_run = []
        print(f"Option 1 ('paper / human review'): Iterating over 'paper_content' with no LLM parameters for grouping.", flush=True)
    else:
        # For other options, LLM parameters are relevant for grouping
        df_to_use_for_iteration = merged_df.copy()
        current_param_cols_for_run = PARAM_COLS
        print(f"Option {TVD_MI_SCORE_OPTION}: Iterating over 'merged_df' with parameters: {current_param_cols_for_run} for grouping.", flush=True)


### --- SUBSET FOR TEST RUN ONLY ---
    # Apply grouping and sampling if NUM_ROWS_PER_GROUP is a positive integer
    if NUM_ROWS_PER_GROUP > 0 and current_param_cols_for_run:
        initial_rows = len(df_to_use_for_iteration)
        print(f"Applying sampling: Grouping by {current_param_cols_for_run} and taking the first {NUM_ROWS_PER_GROUP} rows per group.", flush=True)
        df_to_use_for_iteration = df_to_use_for_iteration.groupby(current_param_cols_for_run, dropna=False).head(NUM_ROWS_PER_GROUP).reset_index(drop=True)
        print(f"Reduced total processing from {initial_rows} initial rows to {len(df_to_use_for_iteration)} sampled rows.", flush=True)
    elif NUM_ROWS_PER_GROUP > 0 and not current_param_cols_for_run:
        initial_rows = len(df_to_use_for_iteration)
        print(f"Applying sampling: No explicit parameters to group by, taking the first {NUM_ROWS_PER_GROUP} rows from the entire DataFrame.", flush=True)
        df_to_use_for_iteration = df_to_use_for_iteration.head(NUM_ROWS_PER_GROUP).reset_index(drop=True)
        print(f"Reduced total processing from {initial_rows} initial rows to {len(df_to_use_for_iteration)} sampled rows.", flush=True)
### --- SUBSET FOR TEST RUN ONLY ---

    # Initialize keys
    done_keys = set()
    log_combos_count = 0
    result_combos_count = 0

    # Source 1: Load completed combos from the log file
    if os.path.exists(log_path):
        try:
            log_df = pd.read_parquet(log_path)
            # Filter for rows where 'complete' is True (score columns have valid results or the SKIP_MESSAGE)
            completed_log_combos = log_df[(log_df['complete'] == True)]['combo'].tolist()
            log_combos_count = len(completed_log_combos)
            for combo_str in completed_log_combos:
                # Use current_param_cols_for_run for key generation
                done_keys.add(get_unique_combo_key_from_log_entry(combo_str, current_param_cols_for_run))
            print(f"Loaded {log_combos_count} combos marked 'complete' from log file.", flush=True)
        except Exception as e:
            print(f"Error reading existing log parquet file: {e}. No combos loaded from log.", flush=True)
    else:
        print("No existing log file found. No combos loaded from log.", flush=True)

    # Source 2: Load combos with valid results from the result file
    if os.path.exists(result_path):
        try:
            result_df = pd.read_parquet(result_path)
            # Filter for rows where both average scores are either a string representation of a number or the SKIP_MESSAGE
            valid_result_rows = result_df[
                (result_df['tpr_avg_tvd_mi_score'].apply(lambda x: x == SKIP_MESSAGE or is_numeric_string(x) or x == TOTAL_FAILURE_MESSAGE)) &
                (result_df['fpr_avg_tvd_mi_score'].apply(lambda x: x == SKIP_MESSAGE or is_numeric_string(x) or x == TOTAL_FAILURE_MESSAGE))
            ]
            result_combos_count = len(valid_result_rows)

            for _, row in valid_result_rows.iterrows():
                # Add combos to keys using current_param_cols_for_run
                done_keys.add(get_unique_combo_key(row, current_param_cols_for_run))
            print(f"Loaded {result_combos_count} combos with valid results from result file.", flush=True)
        except Exception as e:
            print(f"Error reading existing result parquet file: {e}. No combos loaded from results.", flush=True)
    else:
        print("No existing result file found. No combos loaded from results.", flush=True)

    print(f"Final count of already-completed combos (from log OR results): {len(done_keys)}", flush=True)

    # Adjust grouping logic based on whether there are parameters to group by
    grouped_iterable = []
    num_param_groups = 0

    if not current_param_cols_for_run:
        # No parameters to group by, so treat the entire df_to_use_for_iteration as one group
        # The 'params' in the loop will be None, and 'group_df' will be the entire df_to_use_for_iteration
        grouped_iterable = [(None, df_to_use_for_iteration)]
        num_param_groups = 1
        print(f"Processing as a single group (no explicit parameters).", flush=True)
    else:
        grouped_iterable = df_to_use_for_iteration.groupby(current_param_cols_for_run, dropna=False)
        num_param_groups = len(grouped_iterable)
        print(f"Starting processing for {num_param_groups} parameter configuration groups.", flush=True)

    # Initialize new_rows_count outside the loops for a global count
    new_rows_count = 0

    # Start iteration by group
    global_item_counter = 0
    # Use list(grouped_iterable) to allow tqdm to get total number of groups
    for group_idx, (params, group_df) in enumerate(tqdm(list(grouped_iterable), desc="Parameter Groups")):

        # Assigns current parameter values. For single group (no params), 'params' will be None.
        # If params is a tuple, convert it to a dict based on current_param_cols_for_run
        param_values_dict = dict(zip(current_param_cols_for_run, params)) if params is not None and current_param_cols_for_run else {}

        # Iterate through each row (paper_id) within the current parameter group
        for _, row in tqdm(group_df.iterrows(), total=len(group_df), desc=f"  Papers in Group {group_idx+1}"):

            try:
                # Increment global counter
                global_item_counter += 1

                # Get paper ID
                paper_id = row['paper_id']

                # Get formatted parameter string for printing for the current paper-parameter combo
                _, print_params_str = format_param_strings(paper_id, param_values_dict)

                # Get current key
                current_combo_key = get_unique_combo_key(row, current_param_cols_for_run)

                # Check if current key is already present
                if current_combo_key in done_keys:
                    continue

                # Call the function to process each paper_id
                results_dict, log_dict = process_single_paper(
                    llm_client, row['paper_id'], row, response_a_col_name,
                    response_b_col_name, merged_df, current_param_cols_for_run
                )

                # Create DataFrame for current results and log entry
                df_combo = pd.DataFrame([results_dict])


                df_log_entry = pd.DataFrame([log_dict], columns=LOG_COL)

                if os.path.exists(result_path):
                    # If result file exist, read, concat and save
                    existing_df = pd.read_parquet(result_path)
                    combined_df = pd.concat([existing_df, df_combo], ignore_index=True)
                    combined_df.to_parquet(result_path, index=False)
                else:
                    # Else create a new result file
                    df_combo.to_parquet(result_path, index=False)

                # Update the global counter and add key immediately after successful result save
                new_rows_count += len(df_combo)
                done_keys.add(current_combo_key)

                if os.path.exists(log_path):
                    try:
                        # If log file exists, read, concat, drop duplicates and save
                        existing_log_df = pd.read_parquet(log_path)
                        combined_log_df = pd.concat([existing_log_df, df_log_entry], ignore_index=True)
                        combined_log_df.drop_duplicates(subset=['combo'], inplace=True)
                        combined_log_df.to_parquet(log_path, index=False)

                    except Exception as e:
                        print(f"Warning: Could not append to existing log parquet file {log_path}: {e}. Skipping log entry for this combo.", flush=True)
                else:
                    # Else write a new log file
                    df_log_entry.to_parquet(log_path, index=False)

            except Exception as e:
                # In case of failure do not save anything and print this error message
                # Use paper_id as identifier since current_combo_key might not be fully formed yet
                error_combo_identifier = f"paper_id={paper_id}" if 'paper_id' in locals() else "unknown combo"
                print(f"[ERROR] {error_combo_identifier} -> {type(e).__name__}: {e}", flush=True)


Prompt input will use columns parsed_text and parsed_llm_review_1
Processing and saving results to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi/tvd_mi_scores_paper_parsed_llm_review_1.parquet
Logging successful iterations to: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi/tvd_mi_log_paper_parsed_llm_review_1.parquet
Option 2: Iterating over 'merged_df' with parameters: ['model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'note_taking'] for grouping.
Applying sampling: Grouping by ['model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'note_taking'] and taking the first 98 rows per group.
Reduced total processing from 512 initial rows to 512 sampled rows.
No existing log file found. No combos loaded from log.
No existing result file found. No combos loaded from results.
Final count of already-completed combos (from log OR results): 0
Starting processing for 209 parameter configuration groups.


Parameter Groups:   0%|          | 0/209 [00:00<?, ?it/s]

  Papers in Group 1:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 2:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 3:   0%|          | 0/2 [00:00<?, ?it/s]

...

  Papers in Group 205:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 206:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 207:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 208:   0%|          | 0/2 [00:00<?, ?it/s]

  Papers in Group 209:   0%|          | 0/96 [00:00<?, ?it/s]

In [36]:
# result.parquet-file
tvd_mi_results_df = pd.read_parquet(result_path)

# Check results
display(tvd_mi_results_df.head(5))
display(tvd_mi_results_df.shape)

# log.parquet-file
# Check log file and its columns
if os.path.exists(log_path):
    try:
        tvd_mi_log_df = pd.read_parquet(log_path)
        # Check if the expected 'combo' column exists
        if 'combo' in tvd_mi_log_df.columns:
            print(f"Log file '{log_path}' contains the 'combo' column.")
            display(tvd_mi_log_df[['combo', 'complete']].head(5))
            display(tvd_mi_log_df.shape)
        else:
            print(f"Warning: Log file '{log_path}' exists but does not contain the expected 'combo' column.")
            display(tvd_mi_log_df.head(5))
            display(tvd_mi_log_df.shape)
    except Exception as e:
        print(f"Error reading log parquet file for display: {e}")
else:
    print(f"Log file '{log_path}' does not exist.")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,tpr_response_a,tpr_response_b,tpr_tvd_mi_score_1,tpr_tvd_mi_score_2,tpr_avg_tvd_mi_score,fpr_response_a,fpr_response_b,fpr_tvd_mi_score_1,fpr_tvd_mi_score_2,fpr_avg_tvd_mi_score
0,WoByU5W5te0,gemini-3-flash-preview,NaN,high,,BALANCED,We present a novel framework to regularize Ne...,PLACEHOLDER REVIEW,0.25,LLM call failed after multiple retries,LLM call failed after multiple retries,We present a novel framework to regularize Ne...,PLACEHOLDER REVIEW,1.0,1.0,1.0
1,vuD2xEtxZcj,gemini-3-flash-preview,NaN,high,,BALANCED,"In deep learning, fine-grained N:M sparsity r...",PLACEHOLDER REVIEW,1.0,0.25,0.625,"In deep learning, fine-grained N:M sparsity r...",PLACEHOLDER REVIEW,Skipped cause of invalid input,Skipped cause of invalid input,Skipped cause of invalid input
2,WoByU5W5te0,gemini-3-flash-preview,NaN,high,,EVALUATION,We present a novel framework to regularize Ne...,PLACEHOLDER REVIEW,1.0,0.25,0.625,In this paper we present a practical Bayesian...,PLACEHOLDER REVIEW,Skipped cause of invalid input,Skipped cause of invalid input,Skipped cause of invalid input
3,vuD2xEtxZcj,gemini-3-flash-preview,NaN,high,,EVALUATION,"In deep learning, fine-grained N:M sparsity r...",PLACEHOLDER REVIEW,1.0,0.25,0.625,Autoencoding has been a popular and effective...,PLACEHOLDER REVIEW,1.0,LLM call failed after multiple retries,LLM call failed after multiple retries
4,WoByU5W5te0,gemini-3-flash-preview,NaN,high,,FAITHFUL,We present a novel framework to regularize Ne...,PLACEHOLDER REVIEW,Skipped cause of invalid input,Skipped cause of invalid input,Skipped cause of invalid input,"In deep learning, fine-grained N:M sparsity r...",PLACEHOLDER REVIEW,0.25,0.0,0.125


(512, 16)

Log file '/content/drive/MyDrive/Analytical_flexibility_llm_reviews/tvd_mi/tvd_mi_log_paper_parsed_llm_review_1.parquet' contains the 'combo' column.


,combo,complete
0,paper_id=WoByU5W5te0|model=gemini-3-flash-prev...,True
1,paper_id=vuD2xEtxZcj|model=gemini-3-flash-prev...,True
2,paper_id=WoByU5W5te0|model=gemini-3-flash-prev...,True
3,paper_id=vuD2xEtxZcj|model=gemini-3-flash-prev...,True
4,paper_id=WoByU5W5te0|model=gemini-3-flash-prev...,True


(512, 4)

### References

The main code logic and in parts exact code has been taken from:

Robertson, Z., & Koyejo, S. (2025). Let's Measure Information Step-by-Step: LLM-Based Evaluation Beyond Vibes. *arXiv preprint arXiv:2508.05469.*

The code can be accessed [here](https://github.com/zrobertson466920/llm-peer-prediction/tree/main).